# Periodic KS: polynomial forecasting and numerical diagnostics

This notebook solves the one-dimensional Kuramoto-Sivashinsky (KS) equation

\[
u_t = -u u_x-u_{xx}-u_{xxxx}
\]

on a periodic domain, fits a dense quadratic nodal surrogate using exact spectral
derivatives, and evaluates the learned dynamics on held-out trajectories.

It also contains the project's supporting experiments:

1. spatial-resolution and polynomial-degree selection;
2. new-initial-condition and long-horizon generalization;
3. feature-level magnitude pruning;
4. stepwise quadratic pruning and greedy cubic augmentation;
5. maximal Lyapunov exponent and nearby-trajectory divergence;
6. learned-versus-true equilibrium analysis; and
7. approximate periodic-orbit seed discovery.

Run the notebook from top to bottom. The solver, baseline fit, and held-out evaluation
run by default. Expensive experiments are controlled by switches in the configuration
cell. Generated artifacts are written under `outputs/`.

The original notebook remains unchanged. The explicitly nonworking masked-refit
experiment and the duplicate magnitude-pruning implementation were removed.

The baseline uses unscaled dense least squares with no sparsity threshold. Pruning, cubic augmentation, equilibrium searches and orbit proposals are separate opt-in experiments. Exact spectral derivative labels are used; this is not scalar-observation identification.


## Environment and execution

From the repository root, install `python -m pip install -e ".[notebooks]"`.
Run this notebook from the root or its own directory. Shared code lives in `dpsr/`.
A default run fits and evaluates the baseline. Sweeps, long diagnostics, pruning,
and orbit searches are opt-in in the configuration cell. Generated results go
under `outputs/`; stored proof inputs remain separate in the CAP directories.


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "dpsr/polynomial.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from dpsr.polynomial import (
    PolynomialLibrary, PolynomialModel, LibraryModel, polynomial_exponents,
    polynomial_feature_names, fit_polynomial_vector_field, solve_dense_regression,
)
from dpsr.dynamics import model_rhs, rk4_step, cumulative_error_horizon, nrmse

import json
import time
import warnings
from dataclasses import asdict, dataclass, replace
from itertools import combinations_with_replacement, product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eig
from scipy.optimize import least_squares
from sklearn.decomposition import PCA

OUTPUT_DIR = REPO_ROOT / "outputs/ks_periodic_forecasting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Long-running analyses are opt-in.
RUN_NODE_DEGREE_SWEEP = False
RUN_NEW_IC_SEARCH = False
RUN_LONG_HORIZON = False
RUN_MAGNITUDE_PRUNING = False
RUN_PRUNE_ADD_SWEEP = False
RUN_LYAPUNOV_ANALYSIS = False
RUN_PCA_DIVERGENCE = False
RUN_EQUILIBRIUM_ANALYSIS = False
RUN_PERIODIC_ORBIT_SEARCH = False

np.seterr(divide="ignore", invalid="ignore", over="ignore")
warnings.filterwarnings("ignore", message="Singular matrix.*", category=UserWarning)
warnings.filterwarnings("ignore", message="Sparsity parameter.*", category=UserWarning)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


## 1. Configuration

The default trajectory contains a 100-time-unit burn-in, 70 time units for training,
and 100 time units for held-out testing. The total duration is therefore 270 time units.


In [ ]:
@dataclass(frozen=True)
class KSConfig:
    domain_length: float = 22.0
    output_dt: float = 0.1
    internal_dt: float = 0.025
    total_time: float = 270.0
    burn_in: float = 100.0
    train_time: float = 70.0
    test_horizon: float = 100.0
    alpha: float = 0.0
    horizon_tolerance: float = 0.5
    max_abs_factor: float = 8.0
    random_seed: int = 1


CONFIG = KSConfig()

# Configuration selected from the original node/degree sweep.
BASELINE_NODES = 14
BASELINE_DEGREE = 2
BASELINE_SEED = 1

print(json.dumps(asdict(CONFIG), indent=2))


## 2. Fourier pseudospectral ETDRK4 solver

The solver uses a real Fourier transform, Cox-Matthews ETDRK4 time stepping, and
3/2 zero-padding for the quadratic nonlinear term. Internal substeps of 0.025 time
units are saved every 0.1 time units by default.


In [ ]:
from dpsr.systems import (
    ks_initial_condition, rfft_wavenumbers, dealiased_nonlinear_rfft,
    ks_rhs_spectral_state, ks_rhs_spectral_batch, simulate_ks_etdrk4,
    simulate_ks_nodes,
)


## 3. Data generation and spectral diagnostic

The burn-in and training spectra provide a qualitative check that the large-scale
spatial content has settled. Their agreement is useful evidence, but it is not by itself
a proof that all transient behavior has disappeared.


In [ ]:
x_default, t_default, X_default = simulate_ks_nodes(
    BASELINE_NODES, CONFIG, seed=BASELINE_SEED
)

burn_index = int(round(CONFIG.burn_in / CONFIG.output_dt))
train_length = int(round(CONFIG.train_time / CONFIG.output_dt)) + 1
test_length = int(round(CONFIG.test_horizon / CONFIG.output_dt)) + 1

X_burn = X_default[:burn_index]
X_train = X_default[burn_index : burn_index + train_length]
X_test = X_default[
    burn_index + train_length - 1 :
    burn_index + train_length - 1 + test_length
]
t_test = np.arange(len(X_test)) * CONFIG.output_dt

def mean_spatial_power_spectrum(states):
    return np.mean(np.abs(np.fft.rfft(states, axis=1)) ** 2, axis=0)

frequencies = np.fft.rfftfreq(
    BASELINE_NODES, d=CONFIG.domain_length / BASELINE_NODES
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(
    frequencies,
    mean_spatial_power_spectrum(X_burn),
    "r--",
    label="Burn-in segment",
)
ax.semilogy(
    frequencies,
    mean_spatial_power_spectrum(X_train),
    "k-",
    label="Training segment",
)
ax.set(
    xlabel="Spatial frequency",
    ylabel="Mean power",
    title="KS spatial power spectrum",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ks_power_spectrum_check.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Training shape: {X_train.shape}")
print(f"Held-out shape: {X_test.shape}")


## 4. Shared polynomial regression, rollout, and metric utilities

All experiments below use exact spectral derivatives for fitting, one RK4 rollout
implementation, one cumulative spatial-error horizon, and one NRMSE definition.
A vectorized polynomial evaluator avoids repeated feature-library overhead.


In [ ]:
def fit_ks_polynomial_model(states, degree, config):
    """Dense unscaled least squares from exact spectral derivative labels."""
    return fit_polynomial_vector_field(
        states, ks_rhs_spectral_batch(states, config.domain_length),
        degree=degree, regularization=config.alpha, normalize=False,
    )


def rollout_safe(model, initial_state, times, max_abs):
    """RK4 rollout that returns None after divergence or non-finite values."""
    times = np.asarray(times)
    dt = float(times[1] - times[0])
    trajectory = np.zeros((len(times), len(initial_state)))
    trajectory[0] = initial_state
    for step in range(1, len(times)):
        next_state = rk4_step(model, trajectory[step - 1], dt)
        if (
            not np.all(np.isfinite(next_state))
            or np.max(np.abs(next_state)) > max_abs
        ):
            return None
        trajectory[step] = next_state
    return trajectory


def predictive_horizon(true_states, predicted_states, dt, tolerance):
    return cumulative_error_horizon(true_states, predicted_states, dt, tolerance)


def normalized_rmse(true_states, predicted_states):
    return nrmse(true_states, predicted_states)


def coefficient_statistics(model):
    coefficients = np.asarray(model.coefficients())
    active = int(np.count_nonzero(coefficients))
    return {
        "n_terms": int(coefficients.size),
        "n_active": active,
        "sparsity": 1.0 - active / coefficients.size,
        "n_features": int(coefficients.shape[1]),
        "n_active_features": int(
            np.count_nonzero(np.linalg.norm(coefficients, axis=0))
        ),
    }


def evaluate_on_reference(model, true_states, times, config, max_abs):
    prediction = rollout_safe(model, true_states[0], times, max_abs)
    return {
        "prediction": prediction,
        "horizon": predictive_horizon(
            true_states,
            prediction,
            config.output_dt,
            config.horizon_tolerance,
        ),
        "nrmse": normalized_rmse(true_states, prediction),
    }


## 5. Optional node-count and polynomial-degree sweep

The sweep writes a checkpoint CSV after every candidate. It is disabled by default.
The baseline configuration below remains explicit, so later cells do not depend on a
pre-existing local CSV.


In [ ]:
def run_single_configuration(n_nodes, degree, seed, config):
    x, times, states = simulate_ks_nodes(n_nodes, config, seed=seed)
    burn = int(round(config.burn_in / config.output_dt))
    train_count = int(round(config.train_time / config.output_dt)) + 1
    test_count = int(round(config.test_horizon / config.output_dt)) + 1
    training = states[burn : burn + train_count]
    truth = states[burn + train_count - 1 : burn + train_count - 1 + test_count]
    forecast_times = np.arange(len(truth)) * config.output_dt

    fitted = fit_ks_polynomial_model(training, degree, config)
    fast_model = fitted
    max_abs = max(
        50.0,
        config.max_abs_factor * np.max(np.abs(training)),
        2.0 * np.max(np.abs(truth)),
    )
    metrics = evaluate_on_reference(
        fast_model, truth, forecast_times, config, max_abs
    )
    return {
        "seed": seed,
        "n_nodes": n_nodes,
        "degree": degree,
        "horizon": metrics["horizon"],
        "nrmse": metrics["nrmse"],
        **coefficient_statistics(fast_model),
    }


def run_model_sweep(
    config,
    node_counts=(8, 10, 12, 14, 16, 20),
    degrees=(1, 2, 3),
    seeds=(1, 2, 3),
):
    rows = []
    output_csv = OUTPUT_DIR / "ks_sweep_results.csv"
    combinations = list(product(seeds, node_counts, degrees))

    for index, (seed, n_nodes, degree) in enumerate(combinations, start=1):
        started = time.perf_counter()
        try:
            row = run_single_configuration(
                n_nodes, degree, seed, config
            )
            row["status"] = "completed"
            row["error"] = ""
        except Exception as exc:
            row = {
                "seed": seed,
                "n_nodes": n_nodes,
                "degree": degree,
                "horizon": np.nan,
                "nrmse": np.nan,
                "status": "failed",
                "error": str(exc),
            }
        row["elapsed_seconds"] = time.perf_counter() - started
        rows.append(row)
        pd.DataFrame(rows).to_csv(output_csv, index=False)
        print(
            f"[{index}/{len(combinations)}] seed={seed}, N={n_nodes}, "
            f"degree={degree}: horizon={row['horizon']}, status={row['status']}"
        )
    return pd.DataFrame(rows)


if RUN_NODE_DEGREE_SWEEP:
    df_sweep = run_model_sweep(CONFIG)
    sweep_summary = (
        df_sweep.groupby(["n_nodes", "degree"])
        .agg(
            mean_horizon=("horizon", "mean"),
            std_horizon=("horizon", "std"),
            mean_nrmse=("nrmse", "mean"),
            failures=("nrmse", lambda values: values.isna().sum()),
        )
        .reset_index()
    )
    display(sweep_summary)
else:
    print("Model sweep skipped. Set RUN_NODE_DEGREE_SWEEP = True to run.")


## 6. Baseline model and held-out evaluation

The default baseline uses 14 spatial nodes, a degree-two polynomial library, and seed 1.
This choice was retained from the original sweep instead of being hidden behind a local
result file.


In [ ]:
baseline_config = replace(CONFIG, random_seed=BASELINE_SEED)
x_baseline, t_baseline, X_baseline = simulate_ks_nodes(
    BASELINE_NODES, baseline_config, seed=BASELINE_SEED
)

burn = int(round(baseline_config.burn_in / baseline_config.output_dt))
train_count = int(round(baseline_config.train_time / baseline_config.output_dt)) + 1
test_count = int(round(baseline_config.test_horizon / baseline_config.output_dt)) + 1
baseline_training = X_baseline[burn : burn + train_count]
baseline_truth = X_baseline[
    burn + train_count - 1 : burn + train_count - 1 + test_count
]
baseline_times = np.arange(len(baseline_truth)) * baseline_config.output_dt

baseline_fitted = fit_ks_polynomial_model(
    baseline_training, BASELINE_DEGREE, baseline_config
)
baseline_model = baseline_fitted
baseline_max_abs = max(
    50.0,
    baseline_config.max_abs_factor * np.max(np.abs(baseline_training)),
    2.0 * np.max(np.abs(baseline_truth)),
)
baseline_evaluation = evaluate_on_reference(
    baseline_model,
    baseline_truth,
    baseline_times,
    baseline_config,
    baseline_max_abs,
)

best = {
    "seed": BASELINE_SEED,
    "n_nodes": BASELINE_NODES,
    "degree": BASELINE_DEGREE,
    "config": baseline_config,
    "x": x_baseline,
    "training": baseline_training,
    "times": baseline_times,
    "truth": baseline_truth,
    "prediction": baseline_evaluation["prediction"],
    "fitted_model": baseline_fitted,
    "model": baseline_model,
    "max_abs": baseline_max_abs,
}

print(f"Held-out horizon: {baseline_evaluation['horizon']:.2f} t.u.")
print(f"Held-out NRMSE: {baseline_evaluation['nrmse']:.5f}")
print(coefficient_statistics(baseline_model))

print("Training-library diagnostics:", {k: v for k, v in baseline_model.diagnostics.items() if k != "singular_values"})


In [ ]:
def plot_spacetime(result, title_prefix=""):
    true_states = result["truth"]
    prediction = result["prediction"]
    if prediction is None:
        print("The rollout diverged; no held-out figure was produced.")
        return None

    times, x = result["times"], result["x"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    vmin = min(true_states.min(), prediction.min())
    vmax = max(true_states.max(), prediction.max())

    for ax, values, title in (
        (axes[0], true_states, "True KS"),
        (axes[1], prediction, "DPSR surrogate"),
        (axes[2], true_states - prediction, "Error"),
    ):
        kwargs = (
            {"cmap": "coolwarm"}
            if title == "Error"
            else {"cmap": "viridis", "vmin": vmin, "vmax": vmax}
        )
        image = ax.imshow(
            values.T,
            origin="lower",
            aspect="auto",
            extent=[times[0], times[-1], x[0], x[-1]],
            **kwargs,
        )
        ax.set(
            xlabel="Prediction time",
            ylabel="Spatial coordinate",
            title=f"{title_prefix}{title}",
        )
        fig.colorbar(image, ax=ax)
    return fig


def plot_horizon(result, tolerance):
    prediction = result["prediction"]
    if prediction is None:
        return None
    error = np.mean((result["truth"] - prediction) ** 2, axis=1)
    cumulative = np.cumsum(error) * result["config"].output_dt
    horizon = predictive_horizon(
        result["truth"],
        prediction,
        result["config"].output_dt,
        tolerance,
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(result["times"], cumulative, color="darkgreen", lw=2)
    ax.axhline(tolerance, color="crimson", ls="--", label=f"Tolerance = {tolerance}")
    ax.axvline(horizon, color="darkorange", ls=":", label=f"Horizon = {horizon:.2f}")
    ax.set(
        xlabel="Prediction time",
        ylabel="Cumulative mean spatial squared error",
        title="Held-out predictive horizon",
    )
    ax.legend()
    return fig


heldout_figure = plot_spacetime(best, "Held-out: ")
if heldout_figure is not None:
    heldout_figure.savefig(
        OUTPUT_DIR / "ks_heldout_spacetime.png", dpi=200, bbox_inches="tight"
    )
    plt.show()

horizon_figure = plot_horizon(best, baseline_config.horizon_tolerance)
if horizon_figure is not None:
    horizon_figure.savefig(
        OUTPUT_DIR / "ks_heldout_horizon.png", dpi=200, bbox_inches="tight"
    )
    plt.show()


## 7. New-initial-condition generalization

This optional search generates a distinct KS trajectory and looks for a bounded learned
rollout from the same unseen state. It reports every attempted seed rather than silently
selecting a successful example.


In [ ]:
def evaluate_new_initial_condition(best, seed, start_time=50.0):
    config = replace(
        best["config"],
        total_time=start_time + best["config"].test_horizon + 5.0,
        random_seed=seed,
    )
    x, _, states = simulate_ks_nodes(best["n_nodes"], config, seed=seed)
    start = int(round(start_time / config.output_dt))
    count = int(round(config.test_horizon / config.output_dt)) + 1
    truth = states[start : start + count]
    times = np.arange(len(truth)) * config.output_dt
    max_abs = max(best["max_abs"], 2.0 * np.max(np.abs(truth)))
    evaluation = evaluate_on_reference(
        best["model"], truth, times, config, max_abs
    )
    return {
        "seed": seed,
        "start_time": start_time,
        "config": config,
        "x": x,
        "times": times,
        "truth": truth,
        "prediction": evaluation["prediction"],
        "horizon": evaluation["horizon"],
        "nrmse": evaluation["nrmse"],
    }


if RUN_NEW_IC_SEARCH:
    new_ic_rows = []
    new_ic_result = None
    for seed in range(10, 30):
        for start_time in (20.0, 30.0, 40.0, 50.0, 60.0):
            result = evaluate_new_initial_condition(best, seed, start_time)
            new_ic_rows.append(
                {
                    "seed": seed,
                    "start_time": start_time,
                    "horizon": result["horizon"],
                    "nrmse": result["nrmse"],
                    "bounded": result["prediction"] is not None,
                }
            )
            if result["prediction"] is not None:
                new_ic_result = result
                break
        if new_ic_result is not None:
            break

    pd.DataFrame(new_ic_rows).to_csv(
        OUTPUT_DIR / "ks_new_ic_search.csv", index=False
    )
    if new_ic_result is not None:
        figure = plot_spacetime(new_ic_result, "New IC: ")
        figure.savefig(
            OUTPUT_DIR / "ks_new_ic_spacetime.png", dpi=200, bbox_inches="tight"
        )
        plt.show()
        print(
            f"Selected seed={new_ic_result['seed']}, "
            f"start={new_ic_result['start_time']}, "
            f"horizon={new_ic_result['horizon']:.2f}, "
            f"NRMSE={new_ic_result['nrmse']:.4f}"
        )
else:
    print("New-IC search skipped. Set RUN_NEW_IC_SEARCH = True to run.")


## 8. Optional long-horizon comparison

The true trajectory is solved with the same internally substepped ETDRK4 method. The
learned rollout begins where the training window ends.


In [ ]:
if RUN_LONG_HORIZON:
    long_config = replace(CONFIG, total_time=270.0, random_seed=best["seed"])
    x_long, t_long, true_full = simulate_ks_nodes(
        best["n_nodes"], long_config, seed=best["seed"]
    )
    comparison_time = long_config.burn_in + long_config.train_time
    comparison_index = int(round(comparison_time / long_config.output_dt))
    comparison_times = t_long[comparison_index:] - t_long[comparison_index]
    true_comparison = true_full[comparison_index:]
    prediction = rollout_safe(
        best["model"],
        true_comparison[0],
        comparison_times,
        max(best["max_abs"], 2.0 * np.max(np.abs(true_comparison))),
    )
    long_result = {
        "config": long_config,
        "x": x_long,
        "times": comparison_times,
        "truth": true_comparison,
        "prediction": prediction,
    }
    print(
        "Long horizon:",
        predictive_horizon(
            true_comparison,
            prediction,
            long_config.output_dt,
            long_config.horizon_tolerance,
        ),
    )
    figure = plot_spacetime(long_result, "Long comparison: ")
    if figure is not None:
        figure.savefig(
            OUTPUT_DIR / "ks_long_spacetime.png", dpi=200, bbox_inches="tight"
        )
        plt.show()
else:
    print("Long-horizon comparison skipped. Set RUN_LONG_HORIZON = True to run.")


## 9. Feature-level magnitude pruning

The one retained sparsity experiment removes whole polynomial features in ascending
order of coefficient-vector norm. Coefficients are not refit. This is the working method
from the original notebook; the failed entry-level masked-refit experiment was omitted.


In [ ]:
class CoefficientModel:
    """A feature library paired with an explicit coefficient matrix."""

    def __init__(self, feature_library, coefficients):
        self.feature_library = feature_library
        self._coefficients = np.asarray(coefficients).copy()
        self._powers = np.asarray(feature_library.powers_)

    def predict(self, states):
        states = np.atleast_2d(states)
        theta = np.prod(
            states[:, None, :] ** self._powers[None, :, :], axis=2
        )
        return theta @ self._coefficients.T

    def coefficients(self):
        return self._coefficients.copy()


def feature_pruned_model(fitted_model, n_prune):
    coefficients = np.asarray(fitted_model.coefficients()).copy()
    order = np.argsort(np.linalg.norm(coefficients, axis=0))
    coefficients[:, order[:n_prune]] = 0.0
    return CoefficientModel(fitted_model.feature_library, coefficients)


def run_magnitude_pruning(best):
    n_features = best["fitted_model"].coefficients().shape[1]
    counts = [
        count
        for count in (0, 1, 2, 3, 4, 5, 10, 15, 20, 25, 30,
                      40, 50, 60, 70, 80, 90, 100)
        if count < n_features
    ]
    rows, models = [], {}
    for count in counts:
        candidate = feature_pruned_model(best["fitted_model"], count)
        evaluation = evaluate_on_reference(
            candidate,
            best["truth"],
            best["times"],
            best["config"],
            best["max_abs"],
        )
        row = {
            "n_pruned_features": count,
            "horizon": evaluation["horizon"],
            "nrmse": evaluation["nrmse"],
            **coefficient_statistics(candidate),
        }
        rows.append(row)
        models[count] = {
            "model": candidate,
            "prediction": evaluation["prediction"],
        }
        print(row)
    return pd.DataFrame(rows), models


if RUN_MAGNITUDE_PRUNING:
    df_pruning, pruned_models = run_magnitude_pruning(best)
    df_pruning.to_csv(
        OUTPUT_DIR / "ks_magnitude_pruning_sweep.csv", index=False
    )
    dense_horizon = baseline_evaluation["horizon"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(
        df_pruning["n_pruned_features"],
        df_pruning["horizon"],
        "o-",
        color="darkgreen",
    )
    axes[0].axhline(
        dense_horizon, color="gray", ls="--", label="Dense baseline"
    )
    axes[0].set(
        xlabel="Pruned library features",
        ylabel="Predictive horizon",
        title="Horizon",
    )
    axes[0].legend()
    axes[1].plot(
        df_pruning["n_pruned_features"],
        df_pruning["nrmse"],
        "s-",
        color="crimson",
    )
    axes[1].set(
        xlabel="Pruned library features",
        ylabel="Held-out NRMSE",
        title="Rollout error",
    )
    axes[2].plot(
        df_pruning["n_active_features"],
        df_pruning["horizon"],
        "D-",
        color="steelblue",
    )
    axes[2].invert_xaxis()
    axes[2].set(
        xlabel="Active library features",
        ylabel="Predictive horizon",
        title="Parsimony-horizon tradeoff",
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "ks_magnitude_pruning_sweep.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Pruning sweep skipped. Set RUN_MAGNITUDE_PRUNING = True to run.")


## 10. Stepwise quadratic pruning and greedy cubic augmentation

This experiment mirrors the two-phase Mackey-Glass sweep. Phase A removes the
active quadratic monomial with the smallest coefficient-vector norm and refits all
remaining coefficients. Phase B starts from the pruned model and uses the same
explicit forward-selection procedure as the Mackey-Glass experiment. At each step,
every remaining cubic monomial is added in turn, all active coefficients are refit,
and the candidate with the smallest training derivative residual is retained.

The first addition tests all 560 cubic candidates; each later addition tests all
remaining unused candidates. Metrics are averaged over the original held-out segment
and five independently generated KS trajectories.


In [ ]:
def build_polynomial_exponents(state_dimension, max_degree):
    exponents, degrees = [], []
    for degree in range(max_degree + 1):
        for combination in combinations_with_replacement(
            range(state_dimension), degree
        ):
            exponent = np.zeros(state_dimension, dtype=int)
            for index in combination:
                exponent[index] += 1
            exponents.append(exponent)
            degrees.append(degree)
    return np.asarray(exponents), np.asarray(degrees)


def polynomial_transform(states, exponents):
    states = np.atleast_2d(states)
    return np.prod(
        states[:, None, :] ** exponents[None, :, :], axis=2
    )


def polynomial_feature_names(exponents):
    names = []
    for exponent in exponents:
        factors = []
        for index, power in enumerate(exponent):
            if power == 1:
                factors.append(f"x{index}")
            elif power > 1:
                factors.append(f"x{index}^{power}")
        names.append("1" if not factors else " ".join(factors))
    return names


class ActivePolynomialModel:
    """Polynomial model that evaluates only its active library columns."""

    def __init__(self, exponents, coefficients):
        self.exponents = np.asarray(exponents)
        self._coefficients = np.asarray(coefficients).copy()
        self._active = np.flatnonzero(
            np.linalg.norm(self._coefficients, axis=1) > 0.0
        )

    def predict(self, states):
        theta = polynomial_transform(
            states, self.exponents[self._active]
        )
        return theta @ self._coefficients[self._active]

    def coefficients(self):
        return self._coefficients.T.copy()


def fit_active_polynomial(theta, derivatives, active_mask):
    active = np.flatnonzero(active_mask)
    fitted, *_ = np.linalg.lstsq(
        theta[:, active], derivatives, rcond=None
    )
    coefficients = np.zeros((theta.shape[1], derivatives.shape[1]))
    coefficients[active] = fitted
    return coefficients


def build_prune_add_cases(
    best, seeds=(10, 11, 12, 13, 14), start_time=50.0
):
    cases = [
        {
            "label": "held_out",
            "truth": best["truth"],
            "times": best["times"],
            "config": best["config"],
            "max_abs": best["max_abs"],
        }
    ]
    for seed in seeds:
        config = replace(
            best["config"],
            total_time=start_time + best["config"].test_horizon + 5.0,
            random_seed=seed,
        )
        _, _, states = simulate_ks_nodes(best["n_nodes"], config, seed=seed)
        start = int(round(start_time / config.output_dt))
        count = int(round(config.test_horizon / config.output_dt)) + 1
        truth = states[start : start + count]
        cases.append(
            {
                "label": f"seed_{seed}",
                "truth": truth,
                "times": np.arange(len(truth)) * config.output_dt,
                "config": config,
                "max_abs": max(
                    best["max_abs"], 2.0 * np.max(np.abs(truth))
                ),
            }
        )
    return cases


def evaluate_prune_add_candidate(model, cases):
    horizons, nrmses = [], []
    for case in cases:
        evaluation = evaluate_on_reference(
            model,
            case["truth"],
            case["times"],
            case["config"],
            case["max_abs"],
        )
        if evaluation["prediction"] is not None:
            horizons.append(evaluation["horizon"])
            nrmses.append(evaluation["nrmse"])
    n_stable = len(horizons)
    return {
        "n_stable": n_stable,
        "n_cases": len(cases),
        "stable_fraction": n_stable / len(cases),
        "mean_horizon": float(np.mean(horizons)) if horizons else np.nan,
        "std_horizon": float(np.std(horizons)) if horizons else np.nan,
        "mean_nrmse": float(np.mean(nrmses)) if nrmses else np.nan,
    }


def run_ks_prune_add_sweep(
    training, config, cases, prune_steps=8, add_steps=8
):
    derivatives = ks_rhs_spectral_batch(training, config.domain_length)
    exponents, degrees = build_polynomial_exponents(
        training.shape[1], max_degree=3
    )
    names = polynomial_feature_names(exponents)
    theta = polynomial_transform(training, exponents)
    quadratic_or_less = degrees <= 2
    quadratic = degrees == 2
    cubic = degrees == 3
    rows, models = [], {}

    def record(stage, step, mask, coefficients, **action):
        candidate = ActivePolynomialModel(exponents, coefficients)
        active_theta = theta[:, mask]
        metrics = evaluate_prune_add_candidate(candidate, cases)
        residual = np.linalg.norm(theta @ coefficients - derivatives)
        row = {
            "step": step,
            "stage": stage,
            "n_active_features": int(mask.sum()),
            "cond_theta": float(np.linalg.cond(active_theta)),
            "relative_train_residual": float(
                residual / (np.linalg.norm(derivatives) + 1e-15)
            ),
            **metrics,
            **action,
        }
        rows.append(row)
        models[step] = candidate
        print(row)

    active_mask = quadratic_or_less.copy()
    coefficients = fit_active_polynomial(theta, derivatives, active_mask)
    record("baseline_quadratic", 0, active_mask, coefficients)

    for step in range(1, prune_steps + 1):
        coefficient_norms = np.linalg.norm(coefficients, axis=1)
        scores = np.where(active_mask & quadratic, coefficient_norms, np.inf)
        weakest = int(np.argmin(scores))
        if not np.isfinite(scores[weakest]):
            break
        active_mask[weakest] = False
        coefficients = fit_active_polynomial(theta, derivatives, active_mask)
        record(
            "prune_quadratic",
            step,
            active_mask,
            coefficients,
            removed_feature=names[weakest],
        )

    remaining_cubic = list(np.flatnonzero(cubic & ~active_mask))
    for addition in range(1, add_steps + 1):
        best_feature = None
        best_residual = np.inf
        best_coefficients = None

        for feature in remaining_cubic:
            trial_mask = active_mask.copy()
            trial_mask[feature] = True
            trial_coefficients = fit_active_polynomial(
                theta, derivatives, trial_mask
            )
            trial_residual = np.linalg.norm(
                theta @ trial_coefficients - derivatives
            )
            if trial_residual < best_residual:
                best_feature = feature
                best_residual = trial_residual
                best_coefficients = trial_coefficients

        if best_feature is None:
            break
        active_mask[best_feature] = True
        remaining_cubic.remove(best_feature)
        coefficients = best_coefficients
        record(
            "add_cubic",
            prune_steps + addition,
            active_mask,
            coefficients,
            added_feature=names[best_feature],
            selected_train_residual=float(best_residual),
        )

    return pd.DataFrame(rows), models


def plot_ks_prune_add_sweep(results):
    baseline = results.iloc[0]
    prune = results[results["stage"] == "prune_quadratic"]
    add = results[results["stage"] == "add_cubic"]
    metrics = [
        ("mean_horizon", "Mean predictive horizon (t.u.)", False),
        ("mean_nrmse", "Mean NRMSE", False),
        ("cond_theta", r"Library condition number $\kappa(\Theta)$", True),
    ]
    fig, axes = plt.subplots(
        1, 3, figsize=(15, 4.6), constrained_layout=True
    )
    for ax, (metric, ylabel, log_scale) in zip(axes, metrics):
        ax.axhline(
            baseline[metric], color="0.55", ls=":", lw=1.5,
            label="Dense quadratic baseline"
        )
        ax.scatter(
            baseline["step"], baseline[metric],
            s=30 + 220 * baseline["stable_fraction"], color="0.35",
            edgecolor="black", lw=0.6, zorder=4
        )
        ax.plot(
            np.r_[baseline["step"], prune["step"]],
            np.r_[baseline[metric], prune[metric]],
            color="forestgreen", lw=2
        )
        ax.scatter(
            prune["step"], prune[metric],
            s=30 + 220 * prune["stable_fraction"], color="forestgreen",
            edgecolor="black", lw=0.5, label="Quadratic pruning", zorder=4
        )
        if len(add):
            phase_b_start = prune.iloc[-1] if len(prune) else baseline
            ax.plot(
                np.r_[phase_b_start["step"], add["step"]],
                np.r_[phase_b_start[metric], add[metric]],
                color="rebeccapurple", lw=2
            )
            ax.scatter(
                add["step"], add[metric],
                s=30 + 220 * add["stable_fraction"],
                color="rebeccapurple", edgecolor="black", lw=0.5,
                label="Greedy cubic addition", zorder=4
            )
        ax.set(xlabel="Sweep step", ylabel=ylabel)
        ax.set_xticks(results["step"])
        if log_scale:
            ax.set_yscale("log")
    axes[0].set_title("Predictive horizon")
    axes[1].set_title("Rollout error")
    axes[2].set_title("Regression conditioning")
    axes[0].legend(fontsize=8)
    fig.suptitle(
        "Marker size indicates the bounded-rollout fraction across six cases",
        fontsize=10,
    )
    return fig


if RUN_PRUNE_ADD_SWEEP:
    prune_add_cases = build_prune_add_cases(best)
    df_prune_add, prune_add_models = run_ks_prune_add_sweep(
        best["training"], best["config"], prune_add_cases
    )
    df_prune_add.to_csv(
        OUTPUT_DIR / "ks_prune_quadratic_add_cubic_sweep.csv", index=False
    )
    if df_prune_add.iloc[0]["relative_train_residual"] < 1e-10:
        print(
            "The dense quadratic model already fits the training derivative at "
            "numerical precision; differences among cubic candidate residuals "
            "should therefore be interpreted as roundoff-scale tie breaking."
        )
    figure = plot_ks_prune_add_sweep(df_prune_add)
    figure.savefig(
        OUTPUT_DIR / "KS_Prune_Add_Sweep.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    fully_stable = df_prune_add[df_prune_add["stable_fraction"] == 1.0]
    if len(fully_stable):
        best_stable_step = fully_stable.loc[
            fully_stable["mean_horizon"].idxmax()
        ]
        print("Best fully stable step:")
        print(best_stable_step.to_string())
else:
    print("Prune/add sweep skipped. Set RUN_PRUNE_ADD_SWEEP = True to run.")


## 11. Maximal Lyapunov exponent

Both estimators evolve two nearby trajectories and renormalize their separation at every
output step. The true-system estimate uses ETDRK4; the learned-system estimate uses RK4.


In [ ]:
def estimate_mle_learned(
    model, initial_state, dt=0.1, duration=80.0, epsilon=1e-7, max_abs=100.0
):
    local_rng = np.random.default_rng(0)
    direction = local_rng.normal(size=len(initial_state))
    direction /= np.linalg.norm(direction)
    reference = initial_state.copy()
    perturbed = reference + epsilon * direction
    log_sum, elapsed = 0.0, 0.0
    times, running = [], []

    for _ in range(int(round(duration / dt))):
        reference = rk4_step(model, reference, dt)
        perturbed = rk4_step(model, perturbed, dt)
        if (
            not np.all(np.isfinite(reference))
            or not np.all(np.isfinite(perturbed))
            or np.max(np.abs(reference)) > max_abs
        ):
            break
        delta = perturbed - reference
        distance = np.linalg.norm(delta)
        if distance <= 0 or not np.isfinite(distance):
            break
        log_sum += np.log(distance / epsilon)
        elapsed += dt
        perturbed = reference + epsilon * delta / distance
        times.append(elapsed)
        running.append(log_sum / elapsed)
    return (
        log_sum / elapsed if elapsed else np.nan,
        np.asarray(times),
        np.asarray(running),
    )


def estimate_mle_true(
    initial_state, domain_length, dt=0.1, duration=80.0, epsilon=1e-7
):
    local_rng = np.random.default_rng(0)
    direction = local_rng.normal(size=len(initial_state))
    direction /= np.linalg.norm(direction)
    reference = initial_state.copy()
    perturbed = reference + epsilon * direction
    log_sum, elapsed = 0.0, 0.0
    times, running = [], []

    for _ in range(int(round(duration / dt))):
        reference = simulate_ks_etdrk4(
            reference, domain_length, dt, 1
        )[1]
        perturbed = simulate_ks_etdrk4(
            perturbed, domain_length, dt, 1
        )[1]
        delta = perturbed - reference
        distance = np.linalg.norm(delta)
        if distance <= 0 or not np.isfinite(distance):
            break
        log_sum += np.log(distance / epsilon)
        elapsed += dt
        perturbed = reference + epsilon * delta / distance
        times.append(elapsed)
        running.append(log_sum / elapsed)
    return (
        log_sum / elapsed if elapsed else np.nan,
        np.asarray(times),
        np.asarray(running),
    )


if RUN_LYAPUNOV_ANALYSIS:
    mle_learned, mle_learned_times, mle_learned_running = estimate_mle_learned(
        best["model"], best["truth"][0], max_abs=max(100.0, best["max_abs"])
    )
    mle_true, mle_true_times, mle_true_running = estimate_mle_true(
        best["truth"][0], CONFIG.domain_length
    )
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(
        mle_true_times,
        mle_true_running,
        color="crimson",
        label=f"True KS: {mle_true:.4f}",
    )
    ax.plot(
        mle_learned_times,
        mle_learned_running,
        color="steelblue",
        label=f"Learned: {mle_learned:.4f}",
    )
    ax.axhline(0.043, color="gray", ls=":", label="Literature reference ~0.043")
    ax.axhline(0.0, color="black", lw=0.8)
    ax.set(
        xlabel="Time",
        ylabel="Running MLE estimate",
        title="Maximum Lyapunov exponent",
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "ks_mle.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    mle_true = np.nan
    mle_learned = np.nan
    print("Lyapunov analysis skipped. Set RUN_LYAPUNOV_ANALYSIS = True to run.")


## 12. Nearby-initial-condition divergence in PCA space

PCA is fit only to true KS trajectories. The same projection is then applied to learned
trajectories, ensuring the two panels use one coordinate system.


In [ ]:
if RUN_PCA_DIVERGENCE:
    n_trajectories = 5
    epsilon = 1e-5
    duration = 500.0
    dt = CONFIG.output_dt
    times = np.arange(0.0, duration + dt, dt)
    n_steps = len(times) - 1
    local_rng = np.random.default_rng(7)
    base_state = best["truth"][0].copy()

    initial_conditions = [base_state]
    for _ in range(n_trajectories - 1):
        direction = local_rng.normal(size=len(base_state))
        direction /= np.linalg.norm(direction)
        initial_conditions.append(base_state + epsilon * direction)

    true_trajectories = np.asarray(
        [
            simulate_ks_etdrk4(
                initial, CONFIG.domain_length, dt, n_steps
            )
            for initial in initial_conditions
        ]
    )
    learned_trajectories = [
        rollout_safe(
            best["model"],
            initial,
            times,
            max(50.0, 2.0 * np.max(np.abs(true_trajectories))),
        )
        for initial in initial_conditions
    ]

    pca = PCA(n_components=3).fit(
        true_trajectories.reshape(-1, best["n_nodes"])
    )
    true_pc = np.asarray(
        [pca.transform(trajectory) for trajectory in true_trajectories]
    )
    learned_pc = [
        pca.transform(trajectory) if trajectory is not None else None
        for trajectory in learned_trajectories
    ]

    colors = plt.cm.viridis(np.linspace(0, 1, n_trajectories))
    fig = plt.figure(figsize=(14, 6))
    true_ax = fig.add_subplot(121, projection="3d")
    learned_ax = fig.add_subplot(122, projection="3d")
    for index, color in enumerate(colors):
        true_ax.plot(*true_pc[index].T, color=color, lw=1.0)
        if learned_pc[index] is not None:
            learned_ax.plot(*learned_pc[index].T, color=color, lw=1.0)
    true_ax.set_title("True KS")
    learned_ax.set_title("Learned surrogate")
    for ax in (true_ax, learned_ax):
        ax.set(xlabel="PC1", ylabel="PC2", zlabel="PC3")
    fig.suptitle(
        f"Nearby trajectories; first three PCs explain "
        f"{pca.explained_variance_ratio_[:3].sum():.1%}"
    )
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / "ks_nearby_ic_pca.png", dpi=200, bbox_inches="tight"
    )
    plt.show()
else:
    print("PCA divergence skipped. Set RUN_PCA_DIVERGENCE = True to run.")


## 13. Current-model equilibrium benchmark

This benchmark runs by default on the fitted 14-node quadratic forecasting model.
Both fields receive the same 149 starts (constant, 20 training states, 128 seeded
random states), a fixed solver budget, and the training-mean constraint. Accepted
roots have field residual and mean error below 1e-8. Deduplication and one-to-one
matching use minimum distance over every discrete spatial shift, at tolerance
1e-3. Full coordinates, local spectra and failed attempts are saved to JSON.
Stability counts use real-part tolerance 1e-5. This finite-system diagnostic is
not an exhaustive search or a PDE certificate.


In [ ]:
def finite_difference_jacobian(function, state, epsilon=1e-6):
    jacobian = np.zeros((len(state), len(state)))
    for index in range(len(state)):
        perturbation = np.zeros(len(state))
        perturbation[index] = epsilon
        jacobian[:, index] = (
            function(state + perturbation)
            - function(state - perturbation)
        ) / (2.0 * epsilon)
    return jacobian


def canonical_shift(state):
    return np.roll(state, -int(np.argmax(state)))


def distance_modulo_shift(first, second):
    return min(
        np.linalg.norm(np.roll(first, shift) - second)
        for shift in range(len(first))
    )


def find_equilibria(
    function,
    training_states,
    target_mean,
    name,
    n_random_starts=300,
    time_limit=120.0,
):
    local_rng = np.random.default_rng(42)
    starts = [np.full(training_states.shape[1], target_mean)]
    starts.extend(
        training_states[
            np.linspace(0, len(training_states) - 1, 20, dtype=int)
        ]
    )
    starts.extend(
        local_rng.uniform(
            training_states.min(),
            training_states.max(),
            size=(n_random_starts, training_states.shape[1]),
        )
    )

    roots = []
    started = time.perf_counter()
    for start in starts:
        if time.perf_counter() - started > time_limit:
            break

        def constrained_residual(state):
            return np.r_[
                function(state),
                10.0 * (np.mean(state) - target_mean),
            ]

        try:
            solution = least_squares(
                constrained_residual,
                start,
                xtol=1e-12,
                ftol=1e-12,
                gtol=1e-12,
                max_nfev=5000,
            )
        except Exception:
            continue

        candidate = solution.x
        if (
            solution.success
            and np.linalg.norm(function(candidate)) < 1e-8
            and abs(np.mean(candidate) - target_mean) < 1e-8
            and all(
                np.linalg.norm(
                    canonical_shift(candidate) - canonical_shift(existing)
                )
                >= 1e-3
                for existing in roots
            )
        ):
            roots.append(candidate)

    print(f"{name}: found {len(roots)} distinct roots.")
    return roots


def equilibrium_records(roots, function, training_states, label):
    records = []
    for index, root_state in enumerate(roots):
        jacobian = finite_difference_jacobian(function, root_state)
        eigenvalues, eigenvectors = eig(jacobian)
        records.append(
            {
                "label": label,
                "index": index,
                "state": root_state,
                "residual": float(np.linalg.norm(function(root_state))),
                "eigenvalues": eigenvalues,
                "eigenvectors": eigenvectors,
                "min_training_distance": float(
                    np.min(np.linalg.norm(training_states - root_state, axis=1))
                ),
            }
        )
    return records


# Run independently of the expensive orbit/manifold switches.
from dpsr.equilibria import benchmark_mg, benchmark_ks, save_benchmark, summary_rows

if RUN_EQUILIBRIUM_ANALYSIS:
    ks_equilibrium_benchmark = benchmark_ks(
        best["model"], best["training"],
        lambda state: ks_rhs_spectral_state(state, best["config"].domain_length),
    )
    save_benchmark(ks_equilibrium_benchmark, OUTPUT_DIR / "equilibria")
    display(pd.DataFrame(summary_rows(ks_equilibrium_benchmark)))
    print("One-to-one matches modulo discrete spatial shifts; not an exhaustive root search.")
else:
    print("Equilibrium benchmark skipped. Set RUN_EQUILIBRIUM_ANALYSIS = True to run.")


## 14. Relative periodic-orbit proposal and true-system validation

This pilot scans a long learned trajectory for close returns after quotienting spatial
translation.  It refines the best seeds by symmetry-aware single shooting at fixed
spatial mean, evaluates each refined candidate under the true 14-node KS flow, and
attempts a second correction there.  Seeds and failed corrections are retained.


In [ ]:
def rollout_fixed_steps(model, initial_state, dt, n_steps, max_abs=100.0):
    trajectory = np.zeros((n_steps + 1, len(initial_state)))
    trajectory[0] = initial_state
    for step in range(1, n_steps + 1):
        trajectory[step] = rk4_step(model, trajectory[step - 1], dt)
        if (
            not np.all(np.isfinite(trajectory[step]))
            or np.max(np.abs(trajectory[step])) > max_abs
        ):
            return trajectory[:step], False
    return trajectory, True


def generate_learned_attractor_trajectory(
    model,
    initial_state,
    dt,
    burn_time=200.0,
    sample_time=4000.0,
    max_abs=100.0,
):
    burn, bounded = rollout_fixed_steps(
        model,
        initial_state,
        dt,
        int(round(burn_time / dt)),
        max_abs,
    )
    if not bounded:
        raise RuntimeError("Learned-model burn-in diverged.")
    trajectory, bounded = rollout_fixed_steps(
        model,
        burn[-1],
        dt,
        int(round(sample_time / dt)),
        max_abs,
    )
    if not bounded:
        warnings.warn("Learned attractor trajectory stopped early.")
    return trajectory


def scan_periodic_return_candidates(
    trajectory,
    dt,
    min_period=10.0,
    max_period=300.0,
    keep_per_lag=3,
    max_candidates=100,
):
    scale = np.std(trajectory, axis=0)
    scale[scale < 1e-12] = 1.0
    rows = []

    minimum_lag = int(round(min_period / dt))
    maximum_lag = min(
        int(round(max_period / dt)), len(trajectory) - 2
    )
    for lag in range(minimum_lag, maximum_lag + 1):
        scaled_distance = np.linalg.norm(
            (trajectory[lag:] - trajectory[:-lag]) / scale, axis=1
        )
        count = min(keep_per_lag, len(scaled_distance))
        starts = np.argpartition(scaled_distance, count - 1)[:count]
        for start in starts:
            end = start + lag
            segment = trajectory[start : end + 1]
            return_error = np.linalg.norm(trajectory[end] - trajectory[start])
            diameter = np.max(
                np.linalg.norm(segment - segment[0], axis=1)
            )
            rows.append(
                {
                    "start_index": int(start),
                    "end_index": int(end),
                    "period": float(lag * dt),
                    "return_error": float(return_error),
                    "scaled_return_error": float(scaled_distance[start]),
                    "segment_diameter": float(diameter),
                    "relative_return_error": float(
                        return_error / (diameter + 1e-12)
                    ),
                }
            )

    ranked = pd.DataFrame(rows).sort_values(
        ["scaled_return_error", "return_error"]
    )
    kept = []
    for _, candidate in ranked.iterrows():
        duplicate = any(
            abs(candidate["start_index"] - old["start_index"])
            <= int(round(2.0 / dt))
            and abs(candidate["period"] - old["period"]) <= 2.0
            for old in kept
        )
        if not duplicate:
            kept.append(candidate)
        if len(kept) >= max_candidates:
            break
    return pd.DataFrame(kept).reset_index(drop=True)


if RUN_PERIODIC_ORBIT_SEARCH:
    from dpsr.ks_orbits import (
        adaptive_endpoint,
        orbit_record,
        refine_relative_orbit,
        relative_closure_defect,
        scan_relative_recurrences,
        spectral_shift,
    )

    learned_attractor = generate_learned_attractor_trajectory(
        best["model"],
        best["training"][-1],
        best["config"].output_dt,
        burn_time=100.0,
        sample_time=800.0,
        max_abs=max(100.0, 20.0 * np.max(np.abs(best["training"]))),
    )
    periodic_candidates = scan_relative_recurrences(
        learned_attractor,
        best["config"].output_dt,
        best["config"].domain_length,
        min_period=5.0,
        max_period=50.0,
        search_stride=2,
        neighbors=80,
        max_candidates=30,
    )
    periodic_candidates.to_csv(
        OUTPUT_DIR / "ks_relative_recurrence_candidates.csv", index=False
    )
    print("Top relative-recurrence seeds:")
    print(periodic_candidates.head(10).to_string(index=False))

    learned_rhs = lambda state: model_rhs(best["model"], state)
    true_rhs = lambda state: ks_rhs_spectral_state(
        state, best["config"].domain_length
    )
    learned_endpoint = lambda state, period, _n_steps: adaptive_endpoint(
        state, period, learned_rhs, rtol=1e-9, atol=1e-11, max_step=0.2
    )
    true_endpoint = lambda state, period, _n_steps: adaptive_endpoint(
        state, period, true_rhs, rtol=1e-9, atol=1e-11, max_step=0.2
    )

    orbit_rows = []
    learned_orbit_states = []
    true_orbit_states = []
    for candidate_index, candidate in periodic_candidates.head(5).iterrows():
        reference_state = learned_attractor[int(candidate["start_index"])]
        learned_refinement = refine_relative_orbit(
            reference_state,
            candidate["period"],
            candidate["spatial_shift"],
            best["config"].domain_length,
            learned_endpoint,
            learned_rhs,
            max_step=0.1,
            max_nfev=60,
            defect_tolerance=1e-7,
        )

        true_endpoint_before = true_endpoint(
            learned_refinement.state,
            learned_refinement.period,
            max(20, int(np.ceil(learned_refinement.period / 0.1))),
        )
        true_aligned_before = spectral_shift(
            true_endpoint_before,
            -learned_refinement.spatial_shift,
            best["config"].domain_length,
        )
        orbit_scale = max(
            np.linalg.norm(
                learned_refinement.state
                - np.mean(learned_refinement.state)
            ),
            1e-12,
        )
        true_defect_before = float(
            np.linalg.norm(
                true_aligned_before - learned_refinement.state
            )
            / orbit_scale
        )

        true_refinement = refine_relative_orbit(
            learned_refinement.state,
            learned_refinement.period,
            learned_refinement.spatial_shift,
            best["config"].domain_length,
            true_endpoint,
            true_rhs,
            max_step=0.1,
            max_nfev=60,
            defect_tolerance=1e-7,
        )
        learned_validation_endpoint = adaptive_endpoint(
            learned_refinement.state,
            learned_refinement.period,
            learned_rhs,
            rtol=1e-11,
            atol=1e-13,
            max_step=0.1,
        )
        learned_validation_defect = relative_closure_defect(
            learned_refinement.state,
            learned_validation_endpoint,
            learned_refinement.spatial_shift,
            best["config"].domain_length,
        )
        true_validation_endpoint = adaptive_endpoint(
            true_refinement.state,
            true_refinement.period,
            true_rhs,
            rtol=1e-11,
            atol=1e-13,
            max_step=0.1,
        )
        true_validation_defect = relative_closure_defect(
            true_refinement.state,
            true_validation_endpoint,
            true_refinement.spatial_shift,
            best["config"].domain_length,
        )

        row = {
            "candidate": int(candidate_index),
            "seed_period": float(candidate["period"]),
            "seed_spatial_shift": float(candidate["spatial_shift"]),
            "seed_relative_return_error": float(
                candidate["relative_return_error"]
            ),
            **orbit_record("learned", learned_refinement),
            "learned_validation_defect": learned_validation_defect,
            "true_defect_before_correction": true_defect_before,
            **orbit_record("true", true_refinement),
            "true_validation_defect": true_validation_defect,
            "transfer_period_correction": float(
                true_refinement.period - learned_refinement.period
            ),
            "transfer_state_correction": float(
                np.linalg.norm(
                    true_refinement.state - learned_refinement.state
                )
                / orbit_scale
            ),
        }
        orbit_rows.append(row)
        learned_orbit_states.append(learned_refinement.state)
        true_orbit_states.append(true_refinement.state)
        print(
            f"candidate {candidate_index}: "
            f"seed={candidate['relative_return_error']:.3e}, "
            f"learned={learned_refinement.relative_closure_defect:.3e}, "
            f"true-before={true_defect_before:.3e}, "
            f"true-after={true_refinement.relative_closure_defect:.3e}, "
            f"validated={true_validation_defect:.3e}"
        )

    orbit_results = pd.DataFrame(orbit_rows)
    orbit_results.to_csv(
        OUTPUT_DIR / "ks_orbit_transfer_pilot.csv", index=False
    )
    np.savez(
        OUTPUT_DIR / "ks_orbit_transfer_pilot.npz",
        learned_states=np.asarray(learned_orbit_states),
        true_states=np.asarray(true_orbit_states),
        learned_periods=orbit_results["learned_period"].to_numpy(),
        true_periods=orbit_results["true_period"].to_numpy(),
        learned_shifts=orbit_results["learned_spatial_shift"].to_numpy(),
        true_shifts=orbit_results["true_spatial_shift"].to_numpy(),
    )

    figure, axis = plt.subplots(figsize=(8, 4.8))
    candidate_axis = orbit_results["candidate"]
    axis.semilogy(
        candidate_axis,
        orbit_results["seed_relative_return_error"],
        "o-",
        label="Recurrence seed",
    )
    axis.semilogy(
        candidate_axis,
        orbit_results["learned_closure_defect"],
        "s-",
        label="Refined surrogate orbit",
    )
    axis.semilogy(
        candidate_axis,
        orbit_results["true_defect_before_correction"],
        "^-",
        label="True KS before correction",
    )
    axis.semilogy(
        candidate_axis,
        orbit_results["true_validation_defect"],
        "D-",
        label="True KS after correction",
    )
    axis.set(
        xlabel="Candidate rank",
        ylabel="Relative closure defect",
        title="KS surrogate-to-true relative-orbit pilot",
    )
    axis.legend()
    figure.tight_layout()
    figure.savefig(
        OUTPUT_DIR / "KS_Orbit_Transfer_Pilot.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()
else:
    print(
        "Periodic-orbit search skipped. "
        "Set RUN_PERIODIC_ORBIT_SEARCH = True to run."
    )


## 15. Results summary

Optional metrics are displayed as `not run` until their corresponding switches are
enabled. This keeps the summary cell safe after a normal default run.


In [ ]:
baseline_stats = coefficient_statistics(best["model"])

print("=" * 64)
print("KURAMOTO-SIVASHINSKY RESULTS SUMMARY")
print("=" * 64)
print("Solver: ETDRK4, Fourier pseudospectral, 3/2 dealiasing")
print(f"Domain length: {CONFIG.domain_length}")
print(f"Burn-in: {CONFIG.burn_in} t.u.")
print(f"Training duration: {CONFIG.train_time} t.u.")
print(f"Regression nodes: {best['n_nodes']}")
print(f"Polynomial degree: {best['degree']}")
print(f"Active coefficients: {baseline_stats['n_active']} / {baseline_stats['n_terms']}")
print(f"Held-out NRMSE: {baseline_evaluation['nrmse']:.5f}")
print(f"Held-out horizon: {baseline_evaluation['horizon']:.2f} t.u.")
print(
    "True KS MLE:",
    f"{mle_true:.5f}" if np.isfinite(mle_true) else "not run",
)
print(
    "Learned-model MLE:",
    f"{mle_learned:.5f}" if np.isfinite(mle_learned) else "not run",
)


## Repository notes

- Keep generated figures, CSV files, and NPZ files under `outputs/`.
- Interpret long-time pointwise disagreement in the context of chaotic divergence.
- Compare candidate models on the same held-out trajectory and tolerance.
- The periodic-return candidates require a separate Newton or shooting solve before
  they can be called periodic orbits.
- The equilibrium search is stochastic and time-limited; record its settings with any
  reported results.
